# Edges vs. Coverage

Plots the number of edges (x-axis) against the coverage achieved (y-axis), the inverse view of the degree-vs-coverage figure.

Reads the `edge_to_coverage_stats_{dataset}.csv` files written by `edge_to_coverage_analysis.py`. Columns:
`dataset, metric, method, dimensions, sources, total points, edges, mean coverage, median coverage, min coverage, max coverage`.

In [ ]:
import glob
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
# Load every per-dataset stats file into one long DataFrame.
# Adjust the glob if the CSVs live elsewhere.
STATS_GLOB = "edge_to_coverage_stats_*.csv"

files = sorted(glob.glob(STATS_GLOB))
if not files:
    raise FileNotFoundError(
        f"No files match {STATS_GLOB!r}. Run edge_to_coverage_analysis.py first, "
        "or point STATS_GLOB at the directory holding the CSVs."
    )

df = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)
df['edges'] = df['edges'].astype(int)
for col in ['mean coverage', 'median coverage', 'min coverage', 'max coverage']:
    df[col] = df[col].astype(float)
print(f"Loaded {len(files)} file(s): {[os.path.basename(f) for f in files]}")
df.head()

In [ ]:
def plot_edges_vs_coverage(df, stat='mean coverage', show_band=True, save_path=None):
    """One subplot per dataset: edges (x) vs coverage %% (y).

    A separate curve is drawn per (metric, method) combination present for the
    dataset. When `show_band`, the min/max coverage range is shaded behind the
    chosen central `stat` (mean or median coverage).
    """
    datasets = sorted(df['dataset'].unique())
    n_ds = len(datasets)

    fig, axes = plt.subplots(
        1, n_ds, figsize=(5.5 * n_ds, 4.5), squeeze=False, sharey=True
    )

    for c, dataset in enumerate(datasets):
        ax = axes[0, c]
        sub = df[df['dataset'] == dataset]

        for (metric, method), grp in sub.groupby(['metric', 'method']):
            grp = grp.sort_values('edges')
            label = f"{method}" + (f" ({metric})" if str(metric) not in ('', 'nan') else "")
            line, = ax.plot(grp['edges'], grp[stat], marker='o', markersize=3, label=label)
            if show_band:
                ax.fill_between(
                    grp['edges'], grp['min coverage'], grp['max coverage'],
                    color=line.get_color(), alpha=0.12, linewidth=0,
                )

        ax.set_title(dataset, fontweight='bold')
        ax.set_xlabel('Number of edges')
        if c == 0:
            ax.set_ylabel(f'Coverage (%%) \u2014 {stat}')
        ax.set_ylim(0, 101)
        ax.grid(True, alpha=0.3)
        ax.legend(fontsize=8)

    fig.tight_layout()
    if save_path:
        fig.savefig(save_path, dpi=300, bbox_inches='tight')
        print(f"Saved -> {save_path}")
    plt.show()


plot_edges_vs_coverage(df, stat='mean coverage', show_band=True)

In [ ]:
# Single-axes overlay: every dataset on one plot (mean coverage), for quick comparison.
fig, ax = plt.subplots(figsize=(7, 5))
for (dataset, method), grp in df.groupby(['dataset', 'method']):
    grp = grp.sort_values('edges')
    ax.plot(grp['edges'], grp['mean coverage'], marker='o', markersize=3,
            label=f"{dataset} / {method}")

ax.set_xlabel('Number of edges')
ax.set_ylabel('Mean coverage (%)')
ax.set_ylim(0, 101)
ax.grid(True, alpha=0.3)
ax.legend(fontsize=8)
ax.set_title('Edges vs. coverage')
fig.tight_layout()
plt.show()